In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable



In [0]:
%run /Workspace/Users/mohittaneja222@gmail.com/1_dev_food_chain_project/1_Setup/utilities

In [0]:
print(bronze_schema)

In [0]:
dbutils.widgets.text("catalog", "goods_ct", "Catalog")

dbutils.widgets.text("data_source", "customers", "Data Source")
#dbutils.widgets.text("data_source", "products", "Data Source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#print(catalog, data_source)

base_path = f'/Volumes/goods_ct/bronze/vol_goods/child_company/full_load/{data_source}/{data_source}.csv'
print(base_path)



In [0]:
df = (
      spark.read.format("csv").option("header", "true")
                .option("inferSchema", "true")
                .load(base_path)
                .withColumn("read_timestamp",F.current_timestamp())
                .select("*","_metadata.file_name","_metadata.file_size")
)
display(df.limit(10))


In [0]:
df.printSchema()

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

##Silver Processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

In [0]:
df_bronze.printSchema()

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
display(df_duplicates)

In [0]:
print("Rows before duplicates dropped:", df_bronze.count())
df_silver = df_bronze.dropDuplicates(['customer_id'])
print("Rows after duplicates dropped:", df_silver.count())

In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.trim(F.col("customer_name"))
)
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver.select('city').distinct().show()

In [0]:
city_mapping = {
    'Bengaluruu': 'Bengaluru',
    'Bengalore': 'Bengaluru',
    'Hyderabadd' : 'Hyderabad',
    'Hyderbad' : 'Hyderabad',
    'NewDelhi': 'New Delhi',
    'NewDelhee': 'New Delhi',
    'NewDehli': 'New Delhi'
}

allowed = ['Bengaluru', 'Hyderabad', 'New Delhi']

df_silver = (
    df_silver
        .replace(city_mapping, subset='city')
        .withColumn(
            "city",
            F.when(F.col("city").isNull(), None)
             .when(F.col("city").isin(allowed), F.col("city"))
             .otherwise(F.lit(None))
        )
)

df_silver.select('city').distinct().show()

In [0]:
df_silver.select("customer_name").distinct().count()
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(), None)
        .otherwise(F.initcap(F.col("customer_name")))
)
df_silver.select("customer_name").distinct().show(truncate=False)

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
null_customer_names = ['Sprintx Nutrition', 'Zenathlete Foods', 'Primefuel Nutrition', 'Recovery Lane']
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show(truncate=False)

In [0]:
#Correcting Data with Validation from Business Team 

customer_city_fix = {
    #Sprintx Nutrition
    789403: "New Delhi",

    #Zenathlete Foods  
    789420: "Bengaluru",

    #Primefuel Nutrition
    789521: "Hyderabad",

    #Recovery Lane
    789603: "Hyderabad"
}

df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ["customer_id", "fixed_city"]
)

df_fix.display()

In [0]:
df_silver = (
    df_silver
        .join(df_fix, "customer_id", "left")
        .withColumn(
            "city",
            F.coalesce("city", "fixed_city")
        )
        .drop("fixed_city")
)

df_silver.display()

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
df_silver.printSchema()


In [0]:
df_silver = (
    df_silver
    # Build final customer column "CustomerName-City" or "CustomerName - Unknown"

    .withColumn(
        "customer",
           F.concat_ws("-","customer_name", F.coalesce(F.col("city"),F.lit("Unknown")))
    )

# Static attributes aligned with parent data model
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

df_silver.display()

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
            .option("mergeSchema", "true")\
                .mode("overwrite")\
                    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

##Gold processing

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")

#Fetch Required Columns

df_gold = df_silver.select("customer_id", "customer_name", "city", "customer", "market", "platform", "channel")

In [0]:
df_gold.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed", "true")\
            .mode("overwrite")\
                .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
delta_table = DeltaTable.forName(spark, "goods_ct.gold.dim_customers")
df_child_customers = spark.table("goods_ct.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel"
)

In [0]:
delta_table.alias("target").merge(
    source=df_child_customers.alias("source"),
    condition="target.customer_code = source.customer_code"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()